# Driver Drowsiness Detection using Eye Closure and Yawning Analysis

**Methods:** Custom CNN and MobileNetV2 transfer learning  
**Classes:** Closed, Open, no_yawn, yawn  
**Output:** 3-level fatigue state (Alert, Mild Fatigue, Severe Fatigue) and a simulated progression curve.

> Run in Google Colab. Select **Runtime → Change runtime type → T4 GPU** if available. This notebook creates train/validation/test splits from the supplied ZIP because the supplied archive contains only a `train/` directory. The progression curve uses ordered model predictions from held-out images as a *simulation*, not genuine time-series driving footage.

## 1. Upload and extract the dataset

In [ ]:
from google.colab import files
import zipfile, os, glob, pathlib, shutil

uploaded = files.upload()  # Select archive (2).zip
zip_path = next((k for k in uploaded if k.lower().endswith('.zip')), None)
assert zip_path is not None, "Please upload the dataset ZIP file."
EXTRACT_DIR = "/content/driver_drowsiness_data"
if os.path.exists(EXTRACT_DIR): shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(zip_path) as z: z.extractall(EXTRACT_DIR)

# Find the folder containing the four class directories
expected = {"Closed", "Open", "no_yawn", "yawn"}
data_root = None
for root, dirs, files_ in os.walk(EXTRACT_DIR):
    if expected.issubset(set(dirs)):
        data_root = root; break
assert data_root, f"Could not find all four class folders: {expected}"
print("Dataset root:", data_root)

## 2. Dataset inspection and EDA

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd

valid_ext = (".jpg", ".jpeg", ".png", ".bmp")
records = []
for label in sorted(expected):
    for p in glob.glob(os.path.join(data_root, label, "**", "*"), recursive=True):
        if p.lower().endswith(valid_ext): records.append((p, label))
df = pd.DataFrame(records, columns=["filepath", "label"])
print("Total images:", len(df))
print(df["label"].value_counts())
assert set(df.label.unique()) == expected

ax = df.label.value_counts().reindex(sorted(expected)).plot(kind="bar", figsize=(7,4), title="Image count per class")
ax.set_xlabel("Class"); ax.set_ylabel("Number of images"); plt.xticks(rotation=0); plt.show()

from PIL import Image
fig, axes = plt.subplots(1, 4, figsize=(13,3))
for ax, label in zip(axes, sorted(expected)):
    sample = df[df.label == label].iloc[0].filepath
    ax.imshow(Image.open(sample).convert("RGB")); ax.set_title(label); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Train / validation / test split and preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np, tensorflow as tf, random, time

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
IMG_SIZE = (224, 224); BATCH_SIZE = 32
le = LabelEncoder()
df["label_id"] = le.fit_transform(df["label"])
class_names = list(le.classes_)
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df.label_id, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df.label_id, random_state=SEED)
print("Classes / IDs:", dict(enumerate(class_names)))
print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_ds(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame.filepath.values, frame.label_id.values))
    if training: ds = ds.shuffle(len(frame), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_df, True); val_ds = make_ds(val_df); test_ds = make_ds(test_df)


## 4. Custom CNN model

In [ ]:
from tensorflow.keras import layers, models

augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="data_augmentation")

def build_custom_cnn(num_classes):
    inputs = layers.Input(shape=(224,224,3))
    x = augmentation(inputs)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs, outputs, name="Custom_CNN")
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

cnn_model = build_custom_cnn(len(class_names))
cnn_model.summary()
start = time.time()
cnn_history = cnn_model.fit(train_ds, validation_data=val_ds, epochs=5, verbose=1)
cnn_seconds = time.time() - start


## 5. MobileNetV2 transfer learning

In [ ]:
def build_mobilenet(num_classes):
    base = tf.keras.applications.MobileNetV2(input_shape=(224,224,3), include_top=False, weights="imagenet")
    base.trainable = False  # freeze pretrained feature extractor
    inputs = layers.Input(shape=(224,224,3))
    x = augmentation(inputs)
    x = layers.Rescaling(2.0, offset=-1.0)(x)  # MobileNetV2 expects [-1, 1]
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs, name="MobileNetV2_TransferLearning")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

mobile_model = build_mobilenet(len(class_names))
mobile_model.summary()
start = time.time()
mobile_history = mobile_model.fit(train_ds, validation_data=val_ds, epochs=3, verbose=1)
mobile_seconds = time.time() - start


## 6. Training curves and test evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

def plot_history(history, title):
    fig, ax = plt.subplots(1,2, figsize=(11,4))
    ax[0].plot(history.history["accuracy"], label="Train"); ax[0].plot(history.history["val_accuracy"], label="Validation")
    ax[0].set_title(title + " accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()
    ax[1].plot(history.history["loss"], label="Train"); ax[1].plot(history.history["val_loss"], label="Validation")
    ax[1].set_title(title + " loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()

def evaluate_model(model, name):
    y_true = np.concatenate([y.numpy() for _, y in test_ds])
    probs = model.predict(test_ds, verbose=0)
    y_pred = probs.argmax(axis=1)
    print("\n", name, "test accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(xticks_rotation=30, cmap="Blues")
    plt.title(name + " confusion matrix"); plt.show()
    return y_true, y_pred, probs, accuracy_score(y_true, y_pred)

plot_history(cnn_history, "Custom CNN")
cnn_true, cnn_pred, cnn_probs, cnn_acc = evaluate_model(cnn_model, "Custom CNN")
plot_history(mobile_history, "MobileNetV2")
mobile_true, mobile_pred, mobile_probs, mobile_acc = evaluate_model(mobile_model, "MobileNetV2")
comparison = pd.DataFrame({"Model":["Custom CNN", "MobileNetV2"], "Test accuracy":[cnn_acc, mobile_acc], "Training time (seconds)":[cnn_seconds, mobile_seconds]})
display(comparison)


## 7. Four-class prediction to three-level fatigue classification

In [ ]:
# Required mapping: Open/no_yawn -> Alert; yawn -> Mild Fatigue; Closed -> Severe Fatigue
fatigue_map = {"Open": 0, "no_yawn": 0, "yawn": 1, "Closed": 2}
fatigue_names = {0: "Alert", 1: "Mild Fatigue", 2: "Severe Fatigue"}

def four_class_to_fatigue(class_prediction_ids):
    predicted_labels = le.inverse_transform(np.asarray(class_prediction_ids).astype(int))
    return np.array([fatigue_map[label] for label in predicted_labels])

# Use the selected model's held-out test predictions. This does not alter the four-class model.
selected_model = mobile_model if mobile_acc >= cnn_acc else cnn_model
selected_name = "MobileNetV2" if mobile_acc >= cnn_acc else "Custom CNN"
selected_probs = mobile_probs if selected_name == "MobileNetV2" else cnn_probs
selected_pred = selected_probs.argmax(axis=1)
fatigue_stages = four_class_to_fatigue(selected_pred)
print("Selected model:", selected_name)
print(pd.Series([fatigue_names[x] for x in fatigue_stages]).value_counts())


## 8. Fatigue progression curve (simulated session)

In [ ]:
def fatigue_progression(class_prediction_ids, frames_per_minute=30):
    stages = four_class_to_fatigue(class_prediction_ids)
    rows = []
    for start_idx in range(0, len(stages), frames_per_minute):
        chunk = stages[start_idx:start_idx+frames_per_minute]
        # Average severity in each interval; larger values indicate greater fatigue.
        rows.append({"Minute": len(rows)+1, "Mean fatigue score": float(np.mean(chunk)),
                     "Severe frames": int(np.sum(chunk == 2)), "Mild frames": int(np.sum(chunk == 1)),
                     "Alert frames": int(np.sum(chunk == 0))})
    result = pd.DataFrame(rows)
    result["Stage"] = result["Mean fatigue score"].apply(lambda x: "Alert" if x < 0.5 else ("Mild Fatigue" if x < 1.5 else "Severe Fatigue"))
    return result

# IMPORTANT: test images are not time-stamped video frames. This orders held-out predictions
# to demonstrate the requested analysis pipeline; real driving footage is needed for real temporal conclusions.
progress_df = fatigue_progression(selected_pred, frames_per_minute=30)
display(progress_df)
plt.figure(figsize=(10,4))
plt.plot(progress_df["Minute"], progress_df["Mean fatigue score"], marker="o")
plt.yticks([0,1,2], ["Alert", "Mild Fatigue", "Severe Fatigue"]); plt.xlabel("Simulated time (minutes)")
plt.ylabel("Mean fatigue severity"); plt.title("Driver fatigue progression (simulated from test-image predictions)")
plt.grid(True); plt.show()

# Report first observed transitions between adjacent interval labels
stages = progress_df["Stage"].tolist()
for i in range(1, len(stages)):
    if stages[i] != stages[i-1]: print(f"Transition: {stages[i-1]} -> {stages[i]} at simulated minute {i+1}")


## 9. Save models and final outputs

In [ ]:
cnn_model.save("driver_drowsiness_custom_cnn.keras")
mobile_model.save("driver_drowsiness_mobilenetv2.keras")
progress_df.to_csv("fatigue_progression_results.csv", index=False)
comparison.to_csv("model_comparison.csv", index=False)
print("Saved both trained models, model comparison, and progression results.")
from google.colab import files
files.download("driver_drowsiness_custom_cnn.keras")
files.download("driver_drowsiness_mobilenetv2.keras")
files.download("model_comparison.csv")
files.download("fatigue_progression_results.csv")


## 10. Limitations and interpretation
- This dataset contains still images in four classes, not continuous in-car video. The progression plot is therefore a simulation of the requested workflow, not proof of fatigue progression during a real drive.
- A single closed-eye image is not enough to establish prolonged eye closure; a production system should measure closure duration / PERCLOS over time.
- Yawning and eye closure can be affected by lighting, head pose, occlusion, glasses, camera position, and individual differences.
- The dataset split is random and stratified at image level. If multiple images come from the same subject, subject-wise splitting is preferable to reduce identity leakage.
- Model accuracy must be taken from the actual test evaluation above. Do not insert assumed or fabricated accuracy values into the report.
- This is an educational prototype and should not be treated as a certified vehicle safety system.